# 01 · Function Calling
### Practical LLM Engineering — Module 05: LLM Agents

> **Objectives**
> - What function calling is and how it differs from structured prompting
> - Defining typed tool schemas (JSON Schema / OpenAI / Anthropic formats)
> - The function calling request–response–result loop
> - Parallel function calls and sequential chaining
> - Safety guards: blocked tools, rate limits, audit logging
> - Engineering: schema design best practices

---


## 1. Overview

**Function calling** lets an LLM declare it wants to invoke a typed function — returning a structured argument payload that the application executes safely.

```
Without function calling:               With function calling:
────────────────────────                ────────────────────────────────────
User: "What's the weather?"             User: "What's the weather?"
LLM:  "It might be around 15°C"         LLM:  tool_call { get_weather(city="London") }
      [hallucinated guess]              App:  → calls real API → {"temp": 14°C}
                                        LLM:  "It's 14°C and cloudy in London."
```

### Why function calling beats prompt-parsing

| Aspect | Prompt-parsing | Function calling |
|---|---|---|
| Schema enforcement | Soft (model may ignore) | Hard (validated) |
| Type safety | Manual post-processing | Declared in schema |
| Parallel calls | Fragile to parse | Native support |
| Tool selection | Regex-dependent | Model-native |



## 2. Setup

In [ ]:
# !pip install anthropic openai numpy matplotlib -q
import os, re, json, time, math, random, textwrap, traceback
import numpy as np
import matplotlib.pyplot as plt
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional, Any, Callable
from collections import defaultdict

@dataclass
class LLMResponse:
    text: str; input_tokens: int; output_tokens: int; latency_ms: float
    tool_calls: list = field(default_factory=list)
    @property
    def total_tokens(self): return self.input_tokens + self.output_tokens

class MockLLM:
    """Deterministic mock LLM. Replace with Claude/OpenAI client in production."""
    def __init__(self, seed=42):
        self.rng = random.Random(seed)
        self._call_count = 0
    def complete(self, system, user, tools=None, max_tokens=512, temperature=0.0):
        time.sleep(0.02); self._call_count += 1
        text = self._respond(system, user, tools)
        return LLMResponse(text, len((system+user).split()), len(text.split()), 40.0)
    def _respond(self, system, user, tools):
        u = user.lower()
        if tools:
            for t in (tools or []):
                name = t.get("name","")
                desc = t.get("description","").lower()
                kws  = [w for w in desc.split()[:8] if len(w) > 3]
                if any(k in u for k in kws):
                    args = {"query": u[:30], "expression": "sqrt(144)", "city": "Sydney",
                            "code": "print(42)", "n": 3}
                    return json.dumps({"tool_call": {"name": name, "arguments": args}})
        return (f"Based on the provided information: {user[:60].strip()}... "
                "The answer follows from careful analysis.")
    def __call__(self, system, user, **kw):
        return self.complete(system, user, **kw)

llm = MockLLM(seed=42)
print("✅ Shared utilities ready")


## 3. Defining Tool Schemas

In [ ]:
TOOLS = [
    {
        "name": "get_weather",
        "description": "Get current weather conditions for a city.",
        "input_schema": {
            "type": "object",
            "properties": {
                "city":    {"type": "string",  "description": "City name, e.g. 'London'"},
                "country": {"type": "string",  "description": "ISO 3166-1 alpha-2 code"},
                "units":   {"type": "string",  "enum": ["celsius", "fahrenheit"]},
            },
            "required": ["city"],
        },
    },
    {
        "name": "search_web",
        "description": "Search the web for current information on any topic.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query":       {"type": "string",  "description": "Search query"},
                "max_results": {"type": "integer", "description": "Max results (default 5)"},
            },
            "required": ["query"],
        },
    },
    {
        "name": "calculator",
        "description": "Evaluate a mathematical expression. Returns a number.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {"type": "string",
                               "description": "Python math expression, e.g. 'sqrt(144) + 17'"},
            },
            "required": ["expression"],
        },
    },
    {
        "name": "send_email",
        "description": "Send an email to a recipient.",
        "input_schema": {
            "type": "object",
            "properties": {
                "to":      {"type": "string"},
                "subject": {"type": "string"},
                "body":    {"type": "string"},
            },
            "required": ["to", "subject", "body"],
        },
    },
]

def render_tools(tools):
    lines = []
    for t in tools:
        props    = t["input_schema"]["properties"]
        required = t["input_schema"].get("required", [])
        params   = ", ".join(f"{'*' if k in required else ''}{k}:{v['type']}"
                              for k, v in props.items())
        lines.append(f"  {t['name']}({params})\n    → {t['description']}")
    return "\n".join(lines)

print("Available tools (* = required param):\n")
print(render_tools(TOOLS))


## 4. Tool Implementations and Dispatch

In [ ]:
import math as _math

def get_weather(city, country="", units="celsius"):
    db = {"london":{"temp_c":14,"cond":"Cloudy","hum":78},
          "sydney":{"temp_c":22,"cond":"Sunny","hum":55},
          "new york":{"temp_c":8,"cond":"Windy","hum":65},
          "tokyo":{"temp_c":18,"cond":"Clear","hum":60}}
    d = db.get(city.lower(), {"temp_c":20,"cond":"Unknown","hum":50})
    t = d["temp_c"] if units=="celsius" else d["temp_c"]*9/5+32
    return {"city": city, "temperature": f"{t}{'°C' if units=='celsius' else '°F'}",
            "condition": d["cond"], "humidity": f"{d['hum']}%"}

def search_web(query, max_results=5):
    return {"query": query, "results": [
        {"title": f"Result {i+1}: {query}", "url": f"https://example.com/{i+1}",
         "snippet": f"This article covers {query} in detail with practical examples."}
        for i in range(min(max_results, 3))
    ]}

def calculator(expression):
    safe = {k: getattr(_math, k) for k in dir(_math) if not k.startswith("_")}
    safe.update({"abs": abs, "round": round, "min": min, "max": max})
    try:
        result = eval(expression, {"__builtins__": {}}, safe)
        return {"expression": expression, "result": result}
    except Exception as e:
        return {"expression": expression, "error": str(e)}

def send_email(to, subject, body):
    print(f"  [EMAIL SENT] To:{to} Subject:{subject}")
    return {"status": "sent", "to": to, "subject": subject}

REGISTRY = {"get_weather": get_weather, "search_web": search_web,
            "calculator": calculator, "send_email": send_email}

@dataclass
class ToolCallRecord:
    name: str; arguments: dict; result: Any; error: Optional[str]; latency_ms: float

def execute_tool(name, arguments):
    if name not in REGISTRY:
        return ToolCallRecord(name, arguments, None, f"Unknown tool: {name!r}", 0.0)
    t0 = time.perf_counter()
    try:
        result = REGISTRY[name](**arguments); error = None
    except Exception as e:
        result = None; error = str(e)
    return ToolCallRecord(name, arguments, result, error, (time.perf_counter()-t0)*1000)

print("Tool registry ready:", list(REGISTRY.keys()))


## 5. The Function Call Loop

In [ ]:
SYSTEM = ("You are a helpful assistant with tool access. "
          "When you need data, call the appropriate tool. "
          "Respond with JSON: {"tool_call": {"name": "...", "arguments": {...}}}")

def run_tool_call_loop(question, max_turns=3, verbose=True):
    """Full request → tool_call → execute → result → final answer loop."""
    tool_schemas = [{"name": t["name"], "description": t["description"],
                     "parameters": t["input_schema"]} for t in TOOLS]

    if verbose: print(f"Q: {question}")

    for turn in range(max_turns):
        resp = llm(SYSTEM, question, tools=TOOLS, max_tokens=200)

        # Try to parse a tool call
        try:
            parsed = json.loads(resp.text)
            tc     = parsed.get("tool_call", {})
            name   = tc.get("name", "")
            args   = tc.get("arguments", {})
        except (json.JSONDecodeError, AttributeError):
            name, args = "", {}

        if not name or name not in REGISTRY:
            # No valid tool call — treat as final answer
            if verbose: print(f"  [Final] {resp.text[:100]}")
            return resp.text

        # Execute tool
        record = execute_tool(name, args)
        status = "✅" if not record.error else "❌"
        if verbose:
            print(f"  [{status} Tool] {name}({json.dumps(args)[:50]})")
            print(f"  [Result] {str(record.result)[:80]}")

        # Feed result back as context for next turn
        result_str = json.dumps(record.result) if record.result else f"Error: {record.error}"
        question   = f"User asked: {question}\nTool result from {name}: {result_str}\nNow answer:"

    return "Max turns reached."


test_questions = [
    "What is the current weather in Sydney?",
    "What is 15% of 847 plus the square root of 256?",
    "Search for information about HNSW approximate nearest neighbour search.",
]

print("Function Call Loop Demos\n" + "="*50 + "\n")
for q in test_questions:
    run_tool_call_loop(q, verbose=True)
    print()


## 6. Production API Patterns

In [ ]:
# ── Claude tool_use (Anthropic SDK) ──────────────────────────────────
CLAUDE_CODE = '''
import anthropic, json

client = anthropic.Anthropic()

tools = [{
    "name": "get_weather",
    "description": "Get current weather for a city.",
    "input_schema": {
        "type": "object",
        "properties": {"city": {"type": "string"}, "units": {"type": "string", "enum": ["celsius","fahrenheit"]}},
        "required": ["city"]
    }
}]

# Step 1: initial call
response = client.messages.create(
    model="claude-sonnet-4-20250514", max_tokens=1024, tools=tools,
    messages=[{"role": "user", "content": "What is the weather in Tokyo?"}]
)

# Step 2: extract and execute tool call
for block in response.content:
    if block.type == "tool_use":
        result = get_weather(**block.input)          # your function

        # Step 3: return result and get final answer
        followup = client.messages.create(
            model="claude-sonnet-4-20250514", max_tokens=1024, tools=tools,
            messages=[
                {"role": "user",      "content": "What is the weather in Tokyo?"},
                {"role": "assistant", "content": response.content},
                {"role": "user",      "content": [{"type": "tool_result",
                    "tool_use_id": block.id, "content": json.dumps(result)}]},
            ]
        )
        print(followup.content[0].text)
'''

# ── OpenAI function calling ───────────────────────────────────────────
OPENAI_CODE = '''
from openai import OpenAI; import json
client = OpenAI()

tools = [{"type": "function", "function": {
    "name": "get_weather",
    "description": "Get weather for a city.",
    "parameters": {"type": "object",
                   "properties": {"city": {"type": "string"}},
                   "required": ["city"]}
}}]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Weather in Tokyo?"}],
    tools=tools, tool_choice="auto"
)
tc     = response.choices[0].message.tool_calls[0]
result = get_weather(**json.loads(tc.function.arguments))

followup = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Weather in Tokyo?"},
              response.choices[0].message,
              {"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)}],
    tools=tools
)
print(followup.choices[0].message.content)
'''

print("── Anthropic Claude (tool_use) ──")
print(CLAUDE_CODE)
print("── OpenAI (function_calling) ──")
print(OPENAI_CODE)


## 7. Parallel Function Calls

In [ ]:
import threading

@dataclass
class ParallelResult:
    name: str; arguments: dict; result: Any; error: Optional[str]; latency_ms: float

def parallel_tool_calls(tool_calls, timeout_s=5.0):
    """Execute multiple tool calls concurrently via threads."""
    results = [None] * len(tool_calls)
    def run_one(idx, name, arguments):
        rec = execute_tool(name, arguments)
        results[idx] = ParallelResult(rec.name, rec.arguments,
                                       rec.result, rec.error, rec.latency_ms)
    threads = []
    for i, tc in enumerate(tool_calls):
        t = threading.Thread(target=run_one, args=(i, tc["name"], tc["arguments"]))
        threads.append(t); t.start()
    for t in threads: t.join(timeout=timeout_s)
    return results


calls = [
    {"name": "get_weather", "arguments": {"city": "London",   "units": "celsius"}},
    {"name": "get_weather", "arguments": {"city": "Tokyo",    "units": "celsius"}},
    {"name": "get_weather", "arguments": {"city": "New York", "units": "fahrenheit"}},
    {"name": "calculator",  "arguments": {"expression": "22 - 8"}},
]

t0 = time.perf_counter()
presults = parallel_tool_calls(calls)
elapsed  = (time.perf_counter() - t0) * 1000

print(f"Parallel execution: {len(calls)} calls in {elapsed:.0f}ms\n")
for r in presults:
    status = "✅" if not r.error else "❌"
    print(f"  {status} {r.name}({list(r.arguments.values())[0]!r}) → {str(r.result)[:60]}")


## 8. Safe Tool Registry

In [ ]:
class SafeToolRegistry:
    """Wraps tool calls with validation, rate limiting, and audit logging."""
    def __init__(self, registry, rate_limit=10, blocked=None, require_confirm=None):
        self._registry   = registry
        self._rate_limit = rate_limit
        self._counts     = defaultdict(int)
        self._blocked    = set(blocked or [])
        self._confirm    = set(require_confirm or [])
        self._audit      = []

    def call(self, name, arguments, confirmed=False):
        if name in self._blocked:
            return {"error": f"Tool '{name}' is disabled."}
        if name in self._confirm and not confirmed:
            return {"error": f"Tool '{name}' requires confirmation.", "requires_confirmation": True}
        self._counts[name] += 1
        if self._counts[name] > self._rate_limit:
            return {"error": f"Rate limit exceeded for '{name}'."}
        rec = execute_tool(name, arguments)
        self._audit.append({"ts": time.strftime("%H:%M:%S"), "tool": name,
                             "args": arguments, "ok": rec.error is None})
        return rec.result if rec.result else {"error": rec.error}


safe = SafeToolRegistry(REGISTRY, rate_limit=5,
                         blocked=["send_email"], require_confirm=[])

tests = [("get_weather", {"city": "Paris"}),
          ("calculator",  {"expression": "sqrt(144)"}),
          ("send_email",  {"to": "x@y.com", "subject": "Hi", "body": "Hello"}),
          ("unknown",     {})]

print("Safe registry demo:\n")
for name, args in tests:
    result = safe.call(name, args)
    status = "✅" if "error" not in result else "❌"
    print(f"  {status} {name}: {str(result)[:70]}")

print(f"\nAudit log ({len(safe._audit)} entries):")
for e in safe._audit:
    print(f"  [{e['ts']}] {e['tool']} ok={e['ok']}")


## 9. Tool Schema Design Guide

### Best practices

| Aspect | Do | Avoid |
|---|---|---|
| **Name** | `search_products` (verb_noun) | `do_stuff`, `tool1` |
| **Description** | "Returns a list of products with price and stock" | "Gets products" |
| **Required fields** | Only truly mandatory params | Marking everything required |
| **Enums** | `"enum": ["celsius","fahrenheit"]` | Free-form strings for fixed choices |
| **Defaults** | Document in description: "default: 5" | Silent defaults |
| **Granularity** | One clear action per tool | `do_everything(action, params)` |

### Security checklist

| Dangerous tool | Risk | Guard |
|---|---|---|
| `run_shell_command(cmd)` | Arbitrary code execution | Never expose |
| `delete_records(table)` | Irreversible data loss | `require_confirm=True` |
| `send_to_all_users(msg)` | Mass communication | Rate limit + human review |
| `read_file(path)` | Path traversal | Allowlist paths |


